# Push-pull stage (complementary-parallel CS)

In [ ]:
import SLiCAP as sl
sl.initProject("Balanced stages", notebook=True)

## Test setup transadmittance

In [ ]:
push_pull = sl.makeCircuit("kicad/balancing/push_pull.kicad_sch", language="SPICE")
sim_cmd = "OP"
names   = {"Vgs_p": "V(V_P)-V(gp)", 
           "Vgs_n": "V(gn)-V(V_N)", 
           "Id_p": "@M3[id]", 
           "Id_n": "@M1[id]", 
           "Vbias_p": "V(bias_p)", 
           "Vbias_n": "V(bias_n)"}
opinfo  = sl.ngspice2traces("cir/push_pull", sim_cmd, names)
sl.backAnnotateSchematic("kicad/balancing/push_pull.kicad_sch", opinfo)
sl.img2html("push_pull.svg", 700)

### (Almost) odd-order DM transfer

In [ ]:
sim_cmd = "DC V7 -0.5 0.5 10m"
step_cmd = "IQ log 0.1u 10u 3"
names    = {"I_out": "I(V8)"}
DC, x_name, x_units = sl.ngspice2traces("cir/push_pull", sim_cmd, names, stepCmd=step_cmd)
sl.plot("push_pull_dm", "Output current", "lin", 
        DC, xName=x_name, xUnits="V", yScale="u", yUnits="A")
sl.img2html("push_pull_dm.svg", 600)

### (Almost) even-order CM transfer

In [ ]:
sim_cmd = "DC V7 -0.5 0.5 10m"
step_cmd = "IQ log 0.1u 10u 3"
names    = {"I_VP": "-I(V1)", "I_VN": "-I(V2)"}
DC, x_name, x_units = sl.ngspice2traces("cir/push_pull", sim_cmd, names, stepCmd=step_cmd)
sl.plot("push_pull_cm", "Supply current", "lin", 
        DC, xName=x_name, xUnits="V", yScale="u", yUnits="A")
sl.img2html("push_pull_cm.svg", 600)

## Test setup Voltage transfer

In [ ]:
push_pull = sl.makeCircuit("kicad/balancing/push_pull_V.kicad_sch", language="SPICE")
sim_cmd = "OP"
names   = {"Vgs_p": "V(V_P)-V(gp)", 
           "Vgs_n": "V(gn)-V(V_N)", 
           "Id_p": "@M4[id]", 
           "Id_n": "@M2[id]", 
           "Vbias_p": "V(bias_p)", 
           "Vbias_n": "V(bias_n)",
           "V_out": "V(out)",
           "V_P": "V(V1)",
           "V_N": "V(V2)"}
opinfo  = sl.ngspice2traces("cir/push_pull_V", sim_cmd, names)
sl.backAnnotateSchematic("kicad/balancing/push_pull_V.kicad_sch", opinfo)
sl.img2html("push_pull_V.svg", 700)

Save the operating point (10 uA bias current) information for later.

In [ ]:
vgsp   = opinfo["Vgs_p"]
vgsn   = opinfo["Vgs_n"]
VP     = opinfo["V_P"]
VN     = -opinfo["V_N"]
I_bias = (opinfo["Id_n"] + opinfo["Id_p"])/2

In [ ]:
sim_cmd = "DC V7 -0.05 0.05 1m"
step_cmd = "IQ log 0.1u 10u 3"
names    = {"V_out": "V(out)"}
DC, x_name, x_units = sl.ngspice2traces("cir/push_pull_V", sim_cmd, names, stepCmd=step_cmd)
sl.plot("push_pull_V_dm", "Output voltage", "lin", 
        DC, xName=x_name, xUnits="V", yUnits="V")
sl.img2html("push_pull_V_dm.svg", 600)

## Model-based biasing concept

In [ ]:
push_pull_bias_concept = sl.makeCircuit("kicad/balancing/push_pull_bias_concept.kicad_sch", language="SPICE")
sim_cmd = "OP"
names   = {"Id_p": "@M1[id]", 
           "Id_n": "@M2[id]", 
           "V_out": "V(out)"}

# Calculate parameters from bias information:
R_bias  = (VP + VN - vgsp - vgsn)/I_bias
R_P     = (VP - vgsp)/I_bias
R_N     = (VN - vgsn)/I_bias
# Pass parameters to the circuit
params  = [("Vgs_P", vgsp), ("Vgs_N", vgsn), ("R_bias", R_bias), ("R_P", R_P), ("R_N", R_N), ("VP", VP), ("VN", VN)]
opinfo  = sl.ngspice2traces("cir/push_pull_bias_concept", sim_cmd, names, parList=params)
# Add parameters to opinfo for back annotation
opinfo["R_bias"] = R_bias
opinfo["R_P"]    = R_P
opinfo["R_N"]    = R_N
sl.backAnnotateSchematic("kicad/balancing/push_pull_bias_concept.kicad_sch", opinfo)
sl.img2html("push_pull_bias_concept.svg", 600)

In [ ]:
sim_cmd = "DC V7 -0.05 0.05 1m"
names    = {"V_out": "V(out)"}
DC, x_name, x_units = sl.ngspice2traces("cir/push_pull_bias_concept", sim_cmd, names, parList=params)
sl.plot("push_pull_bias_concept_V_dm", "Output voltage", "lin", 
        DC, xName=x_name, xUnits="V", yUnits="V")
sl.img2html("push_pull_bias_concept_V_dm.svg", 600)

In [ ]:
## Model-based biasing implemented

In [ ]:
push_pull_bias = sl.makeCircuit("kicad/balancing/push_pull_bias.kicad_sch", language="SPICE")
sim_cmd = "OP"
names   = {"Id_p": "@M4[id]", 
           "Id_n": "@M2[id]", 
           "V_out": "V(out)"}
opinfo  = sl.ngspice2traces("cir/push_pull_bias", sim_cmd, names, parList=params)
# Add parameters to opinfo for back annotation
opinfo["R_bias"] = R_bias
opinfo["R_P"]    = R_P
opinfo["R_N"]    = R_N
sl.backAnnotateSchematic("kicad/balancing/push_pull_bias.kicad_sch", opinfo)
sl.img2html("push_pull_bias.svg", 600)

In [ ]:
sim_cmd = "DC V7 -0.05 0.05 1m"
names    = {"V_out": "V(out)"}
DC, x_name, x_units = sl.ngspice2traces("cir/push_pull_bias", sim_cmd, names, parList=params)
sl.plot("push_pull_bias_V_dm", "Output voltage", "lin", 
        DC, xName=x_name, xUnits="V", yUnits="V")
sl.img2html("push_pull_bias_V_dm.svg", 600)